# Netflix Recommendation – Machine Learning Report

## Task Basis
The provided task image identifies **Netflix recommendation** as the practical machine-learning task and gives the following ML hierarchy:

1. Data Collection
2. Data Cleaning
3. Data Preprocessing
4. Feature Engineering / Selection
5. Model Selection
6. Model Training
7. Model Evaluation
8. Hyperparameter Tuning
9. Deployment
10. Monitoring & Maintenance

**Important:** The task image does not specify a particular Netflix CSV, recommendation algorithm, evaluation score, or deployment platform. Therefore, this notebook uses a beginner-friendly content-based recommendation implementation as a practical example and does not claim that a specific algorithm was prescribed by the task.

## Tools Used

- Python
- Pandas
- Scikit-learn
- Matplotlib
- Google Colab / Jupyter Notebook

## 1. Data Collection

Upload the Netflix CSV dataset in Google Colab and load it with Pandas.

In [ ]:
import pandas as pd

# In Google Colab, uncomment the following lines to upload your CSV:
# from google.colab import files
# uploaded = files.upload()

# Replace the filename below with the name of your Netflix CSV file.
file_name = 'netflix_titles.csv'
df = pd.read_csv(file_name)

df.head()

## 2. Data Cleaning

Inspect the dataset, check missing values, and remove duplicate records.

In [ ]:
print('Dataset shape:', df.shape)
print('\nColumn names:')
print(df.columns.tolist())

print('\nMissing values:')
print(df.isnull().sum())

print('\nDuplicate rows:', df.duplicated().sum())

df = df.drop_duplicates().copy()
print('Shape after removing duplicates:', df.shape)

## 3. Data Preprocessing

Find commonly used Netflix recommendation fields. The code supports datasets that contain columns such as `title`, `listed_in`, and `description`.

In [ ]:
def find_column(possible_names):
    for name in possible_names:
        if name in df.columns:
            return name
    return None

title_col = find_column(['title', 'Title', 'name', 'Name'])
genre_col = find_column(['listed_in', 'Listed_In', 'genres', 'Genres', 'genre', 'Genre'])
description_col = find_column(['description', 'Description', 'overview', 'Overview'])

print('Title column:', title_col)
print('Genre column:', genre_col)
print('Description column:', description_col)

## 4. Feature Engineering / Selection

For a content-based recommendation example, combine available genre and description information into one text feature.

In [ ]:
if title_col is None:
    raise ValueError('A title column was not found. Please check your Netflix dataset columns.')

text_parts = []
if genre_col is not None:
    text_parts.append(df[genre_col].fillna(''))
if description_col is not None:
    text_parts.append(df[description_col].fillna(''))

if not text_parts:
    raise ValueError('No genre or description column was found for content-based recommendation.')

df['recommendation_text'] = text_parts[0].astype(str)
for part in text_parts[1:]:
    df['recommendation_text'] = df['recommendation_text'] + ' ' + part.astype(str)

df[[title_col, 'recommendation_text']].head()

## 5. Model Selection

The task image does not specify a recommendation algorithm. For this practical notebook, a **content-based recommendation** approach is selected. TF-IDF is used to represent text and cosine similarity is used to find similar titles.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['recommendation_text'])

print('TF-IDF matrix shape:', tfidf_matrix.shape)

## 6. Model Training / Building the Recommendation Matrix

Compute cosine similarity between Netflix titles based on their content features.

In [ ]:
similarity_matrix = cosine_similarity(tfidf_matrix)
print('Similarity matrix shape:', similarity_matrix.shape)

## 7. Model Evaluation

Recommendation tasks do not use the same accuracy metric as ordinary classification. This basic notebook demonstrates recommendation results by inspecting the most similar titles. A production system would require a proper validation strategy and recommendation metrics.

In [ ]:
def recommend(title, n=5):
    matches = df[df[title_col].astype(str).str.lower() == str(title).lower()]
    if matches.empty:
        return f"Title not found: {title}"

    idx = matches.index[0]
    scores = list(enumerate(similarity_matrix[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    top_indices = [i for i, score in scores[1:n+1]]

    result = df.iloc[top_indices][[title_col]].copy()
    result['similarity_score'] = [similarity_matrix[idx][i] for i in top_indices]
    return result.reset_index(drop=True)

# Example: replace this with a title that exists in your dataset.
example_title = df[title_col].dropna().iloc[0]
print('Example title:', example_title)
recommend(example_title, n=5)

## 8. Hyperparameter Tuning

The task image lists hyperparameter tuning as a stage of the ML hierarchy but does not provide specific parameters. For this notebook, the TF-IDF settings can be experimented with, such as `ngram_range`, `min_df`, and `max_features`.

In [ ]:
# Example configuration that can be adjusted:
tfidf_tuned = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1, 2),
    min_df=1
)

tfidf_matrix_tuned = tfidf_tuned.fit_transform(df['recommendation_text'])
similarity_matrix_tuned = cosine_similarity(tfidf_matrix_tuned)

print('Tuned TF-IDF matrix shape:', tfidf_matrix_tuned.shape)

## 9. Deployment

Deployment is listed in the task hierarchy, but the provided task image does not specify a deployment platform. Therefore, deployment is documented as a future step rather than implemented here.

## 10. Monitoring & Maintenance

A deployed recommendation system should be monitored for recommendation quality, changing user preferences, new titles, missing data, and model performance. The provided task image lists this stage but does not give specific monitoring requirements.

## Netflix Recommendation – Conclusion

This notebook demonstrates the machine-learning hierarchy using a Netflix recommendation example. The dataset is inspected and cleaned, recommendation features are prepared, a content-based recommendation approach is built with TF-IDF and cosine similarity, and example recommendations are generated. Deployment and long-term monitoring are identified as future stages.

## Learning References Mentioned in the Task Image

The provided image also mentions:

- Netflix recommendation
- Neural network
- 9 videos / 3 hours
- Learn about CNN Architecture

These items are preserved from the task image. The image does not provide specific video links or a detailed CNN implementation, so no external links or unsupported CNN code are added to this report.